In [1]:
import sys
import numpy as np
import matplotlib as mplt
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import random
import math
import copy
from typing import Annotated, Any, Callable
from pydantic import BaseModel, Field, WithJsonSchema
import pydantic

import cv2

from PIL import Image

from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union


In [2]:
%load_ext autoreload
%autoreload 2

import sys
for p in ['../../../../src']:
    if p not in sys.path:
        sys.path.append(p)
        
import spikeml as sml
from spikeml.utils.nb_util import xdisplay, Markup
from spikeml.core.signal import signal_dc, signal_pulse, encode1_onehot, encode_onehot, signal_ranges, mean_per_input
from spikeml.core.ngram import build_ngram, ngram_find, ngram_msample, print_ngrams

from spikeml.plot.plot_util import plot_hist, plot_data, plot_lidata, plot_input, plot_xt, plot_mt, plot_spikes, imshow_matrix, imshow_nmatrix
from spikeml.core.params import Params, LayerParams, ConnectorParams, SpikeParams, SSensorParams, SNNParams, SSNNParams

from spikeml.core.params import Params, LayerParams, NNParams, ConnectorParams, SpikeParams, SSensorParams, SNNParams, SSNNParams
from spikeml.ui.ipywidgets_ui import ui

from spikeml.core.matrix import matrix_split, normalize_matrix, _mult, cmask, cmask2, matrix_init, matrix_init2
from spikeml.core.monitor import Monitor

from spikeml.core.spikes import pspike, spike
from spikeml.core.base import Component, Module, Fan, Composite, Chain
from spikeml.core.layer import Layer, SimpleLayer, LinearLayer, NormalizeLayer, ThresholdLayer, BinaryThresholdLayer
from spikeml.core.snn import SNN, SSNN, DSSNN
from spikeml.core.sensor import SSensor
from spikeml.core.connector import Connector, LinearConnector, RateConnector, LIConnector, LIConnector2
from spikeml.core.chain import make_snn_chain, make_ssnn_chain, chain_validate

from spikeml.core.feedback import compute_error, xcompute_error, compute_sg, FeedbackAdapter

from spikeml.datasets.dataset import SimpleDataset, DataLoader
from spikeml.core.runner import Context, DataRunner, InferenceRunner, run, nrun, Context, run_with, run_with_feedback


from spikeml.core.sensor_monitor import SSensorMonitor
from spikeml.core.sensor_viewer import SSensorMonitorViewer
from spikeml.core.layer_monitor import LayerMonitor
from spikeml.core.connector_monitor import ConnectorMonitor, LIConnectorMonitor
from spikeml.core.snn_monitor import SNNMonitor, SSNNMonitor
from spikeml.core.layer_viewer import LayerMonitorViewer
from spikeml.core.snn_viewer import SNNMonitorViewer, SSNNMonitorViewer
from spikeml.core.connector_viewer import ConnectorMonitorViewer, LIConnectorMonitorViewer
from spikeml.core.feedback_viewer import ErrorMonitorViewer

from spikeml.core.snn_stats import connector_stats, htest_connections, htest_connector_identity
from spikeml.core.snn_stats_viewer import plot_snn_stats

from spikeml.core.optimize import make_params_spec, vec2dic, setattrs, params_search, set_all_attrs


from spikeml.utils.vector import normalize, normalize_all

from spikeml.utils.img_util import show_img, show_imgs, cv_bgr2rgb

from spikeml.scikit.classifier import ScikitClassifierAdapter


In [3]:
SEED=37
random.seed(SEED)
np.random.seed(SEED)
np.set_printoptions(edgeitems=3, infstr='inf', linewidth=120, nanstr='nan', precision=4, suppress=True, threshold=1000, formatter=None)


In [4]:
from spikeml.datasets.cv.glyph import Glyph, make_glyphs, show_glyphs, make_glyph, glyph_dataset, show_all_glyph_txs

gg = [Glyph.SQUARE, Glyph.CIRC, Glyph.SLASH, Glyph.BACKSLASH]
gg = [Glyph.SLASH, Glyph.BACKSLASH]
xx, yy, labels, dd = glyph_dataset(len(gg), gg=gg, color=(255, 255, 0), unique=True, shuffle=False, shape=(8,8), txs=False)


In [6]:
def make_nn(xx, labels):
    N = len(labels)*4
    nl_1 = NormalizeLayer(scale=3)
    M_1 = RateConnector((N, xx[0].size), params=ConnectorParams(mean=0, sd=.001, t_p=1,t_d=1, cmin=-10, cmax=10))
    M_2 = RateConnector((len(labels), N), params=ConnectorParams(mean=.1, sd=0, t_p=1,t_d=10, cmin=-10, cmax=10))
    ll_1 = LinearLayer(M_1, params=NNParams(), name='ll_1')
    ll_2 = LinearLayer(M_2, params=NNParams(), name='ll_2', force=True) 
    #print(M_1.params.fmt())
    #print(ll_1.params.fmt())
    nn = Chain([nl_1, ll_1, ll_2])
    return nn


xx_ = xx.reshape(xx.shape[0], -1)

nn = make_nn(xx_, labels)
model = ScikitClassifierAdapter(nn, epochs=10, labels=labels, batch_size=-1)
model.fit(xx_, yy)
#show_kernels(model.ref[1].M.M, xx[0].shape)
#show_img(model.ref[1].M.M)
#print(model.ref[1].M.M)


,ref,<spikeml.core...001D6B1696FE0>
,epochs,10
,labels,"[<Glyph.SLASH: 8>, <Glyph.BACKSLASH: 9>]"
,batch_size,-1


In [7]:
X_test = xx_
labels_test = labels
y_test = yy

yy_pred = model.predict(X_test)
print('yy_pred:', yy_pred)
yy_prob = model.predict_proba(X_test)
print('yy_prob:', yy_prob)

score = model.score(X_test, y_test)
print('Accuracy:', score)
for i in range(X_test.shape[0]):
    print(f'{i}: {labels_test[yy[i]]} ({y_test[i]}) -> {yy_prob[i]} ({yy_pred[i]})')



yy_pred: [0 1]
yy_prob: [[1. 0.]
 [0. 1.]]
Accuracy: 1.0
0: Slash (0) -> [1. 0.] (0)
1: Backslash (1) -> [0. 1.] (1)
